# Mixed-form PINN for steady laminar flow past a cylinder

Mixed-variable physics-informed neural network (PINN) for 2D steady incompressible flow past a cylinder in a channel
(the DFG 2D-1 benchmark geometry), following the mixed-form formulation of Rao, Sun & Liu (2020),
*Physics-informed deep learning for incompressible laminar flows*.

**Geometry.** Channel $[0, 1.1] \times [0, 0.41]$, cylinder of radius $0.05$ centred at $(0.2, 0.2)$. $\rho = 1$, $\mu = 0.02$.

**Network.** $(x, y) \mapsto (\psi, p, \sigma_{11}, \sigma_{22}, \sigma_{12})$, 8 hidden layers of 40 neurons, tanh.
Velocity comes from the stream function, $u = \partial\psi/\partial y$, $v = -\partial\psi/\partial x$, so continuity holds exactly.

**Residuals** (only first derivatives of the network outputs are needed):

$$\rho(u u_x + v u_y) - \partial_x\sigma_{11} - \partial_y\sigma_{12} = 0, \qquad
  \rho(u v_x + v v_y) - \partial_x\sigma_{12} - \partial_y\sigma_{22} = 0$$

$$\sigma_{11} = -p + 2\mu u_x, \qquad \sigma_{22} = -p + 2\mu v_y, \qquad \sigma_{12} = \mu(u_y + v_x), \qquad p = -\tfrac{1}{2}(\sigma_{11} + \sigma_{22})$$

**Boundary conditions.** Inlet: parabolic $u = 4U_{max}\,y(0.41 - y)/0.41^2$, $v = 0$; walls and cylinder: no-slip; outlet: $p = 0$.

**Loss.** $\mathcal{L} = \mathcal{L}_{PDE} + 2(\mathcal{L}_{wall} + \mathcal{L}_{inlet} + \mathcal{L}_{outlet})$, trained with Adam then L-BFGS-B.

Files needed in the working directory: `uvNN.pickle` (pretrained weights) and `FluentSol.mat` (ANSYS Fluent reference solution).

### Changes from the original `SteadyFlowCylinder_mixed.py`
The original script targets TensorFlow 1.x and does not run on current versions. Fixes applied:

| Problem in original | Fix |
|---|---|
| `import tensorflow as tf` with TF1 API (`tf.placeholder`, `tf.Session`, `tf.set_random_seed`) | `import tensorflow.compat.v1 as tf` + `tf.disable_v2_behavior()` |
| `tf.contrib.opt.ScipyOptimizerInterface` — `tf.contrib` removed in TF 2.x | Replaced by a small `ScipyLBFGSB` wrapper around `scipy.optimize.minimize` (same options) |
| `from pyDOE import lhs` — module renamed to `pydoe` in pyDOE ≥ 1.5 | Import fallback: `pyDOE` → `pydoe` → `pyDOE3` → SciPy `qmc.LatinHypercube` |
| `np.int` (removed in NumPy 1.24) and `'\pi'` invalid escape | `int(...)` and `r'\pi'` |
| `log_device_placement=True` floods the output | Set to `False` |
| Fluent path `../FluentReferenceMu002/FluentSol.mat` | `FluentSol.mat` in the working directory |
| Five separate `sess.run` calls per Adam step to log losses | One `sess.run` per step (same values, much faster) |
| `save_NN('uvNN.pickle')` overwrites the pretrained weights | Saves to `uvNN_trained.pickle` |
| `postProcess` only saves `uvp.png` | Also displays the figure inline |
| 10,000 Adam steps (lr 5e-4) run after loading the pretrained network, which wrecks it (loss 1.8e-4 → ~6 in 3 steps) | Adam skipped when `LOAD_PRETRAINED = True`; still 10,000 steps when training from scratch |

The physics, network, sampling and loss are unchanged. Added: relative $L_2$ errors against Fluent and a loss-history plot.

## 1. Setup
Requirements: `pip install tensorflow pydoe scipy matplotlib` (tested with TensorFlow 2.21, NumPy 2.4, SciPy 1.17).

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # uncomment to force CPU

import time
import pickle
import random

import numpy as np
import scipy.io
import scipy.optimize
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
tf.logging.set_verbosity(tf.logging.ERROR)

# Latin hypercube sampling: the package has been renamed over time
try:
    from pyDOE import lhs
except ImportError:
    try:
        from pydoe import lhs
    except ImportError:
        try:
            from pyDOE3 import lhs
        except ImportError:
            from scipy.stats import qmc
            def lhs(n, samples):
                return qmc.LatinHypercube(d=n).random(samples)

random.seed(1234)
np.random.seed(1234)
tf.set_random_seed(1234)

print('TensorFlow', tf.__version__ if hasattr(tf, '__version__') else '', '| GPUs:',
      tf.config.list_physical_devices('GPU') if hasattr(tf, 'config') else 'n/a')

## 2. Run configuration
Measured on a 2-core CPU: ~0.9 s per Adam step and ~1.3 s per L-BFGS-B iteration (≈ 50k collocation points),
so the full original schedule (10k Adam + up to 100k L-BFGS-B) is many hours on CPU; use a GPU or lower the counts. Set `RUN_TRAINING = False` to only evaluate the pretrained model, or reduce the iteration counts.

In [ ]:
LOAD_PRETRAINED = True            # False = train from scratch
PRETRAINED_FILE = 'uvNN.pickle'
FLUENT_FILE     = 'FluentSol.mat'
SAVE_FILE       = 'uvNN_trained.pickle'

RUN_TRAINING  = True
# The original runs 10000 Adam steps even when loading the pretrained network. Starting Adam (lr = 5e-4)
# from the converged weights raises the loss from ~1.8e-4 to ~6 within 3 steps, i.e. it discards the
# pretrained solution. So Adam is skipped by default when fine-tuning; L-BFGS-B alone decreases the loss.
ADAM_ITERS    = 0 if LOAD_PRETRAINED else 10000   # original: 10000
ADAM_LR       = 5e-4              # original: 5e-4
BFGS_MAXITER  = 100000            # original: 100000

## 3. L-BFGS-B wrapper (replaces `tf.contrib.opt.ScipyOptimizerInterface`)

In [ ]:
class ScipyLBFGSB:
    """Minimise a TF1-graph loss over a list of variables with scipy's L-BFGS-B."""

    def __init__(self, loss, var_list, options=None):
        self.loss = loss
        self.var_list = var_list
        self.options = options or {}
        self.grads = tf.gradients(loss, var_list)
        self.shapes = [v.get_shape().as_list() for v in var_list]
        self.sizes = [int(np.prod(s)) for s in self.shapes]
        self.placeholders = [tf.placeholder(tf.float32, shape=s) for s in self.shapes]
        self.assign_ops = [v.assign(ph) for v, ph in zip(var_list, self.placeholders)]

    def _set(self, sess, flat):
        feed, i = {}, 0
        for ph, shape, size in zip(self.placeholders, self.shapes, self.sizes):
            feed[ph] = flat[i:i + size].reshape(shape)
            i += size
        sess.run(self.assign_ops, feed)

    def minimize(self, sess, feed_dict, loss_callback=None):
        x0 = np.concatenate([w.ravel() for w in sess.run(self.var_list)]).astype(np.float64)

        def fun(x):
            self._set(sess, x)
            loss, grads = sess.run([self.loss, self.grads], feed_dict)
            return np.float64(loss), np.concatenate([g.ravel() for g in grads]).astype(np.float64)

        def callback(xk):
            if loss_callback is not None:
                self._set(sess, xk)
                loss_callback(sess.run(self.loss, feed_dict))

        res = scipy.optimize.minimize(fun, x0, jac=True, method='L-BFGS-B',
                                      callback=callback, options=self.options)
        self._set(sess, res.x)
        print('L-BFGS-B finished:', res.message, '| iterations:', res.nit, '| final loss: %.3e' % res.fun)
        return res

## 4. Mixed-form PINN model

In [ ]:
class PINN_laminar_flow:
    # Initialize the class
    def __init__(self, Collo, INLET, OUTLET, WALL, uv_layers, lb, ub, ExistModel=0, uvDir='',
                 bfgs_maxiter=100000):

        # Count for callback function
        self.count = 0

        # Bounds
        self.lb = lb
        self.ub = ub

        # Mat. properties
        self.rho = 1.0
        self.mu = 0.02

        # Collocation point
        self.x_c = Collo[:, 0:1]
        self.y_c = Collo[:, 1:2]

        self.x_INLET = INLET[:, 0:1]
        self.y_INLET = INLET[:, 1:2]
        self.u_INLET = INLET[:, 2:3]
        self.v_INLET = INLET[:, 3:4]

        self.x_OUTLET = OUTLET[:, 0:1]
        self.y_OUTLET = OUTLET[:, 1:2]

        self.x_WALL = WALL[:, 0:1]
        self.y_WALL = WALL[:, 1:2]

        # Define layers
        self.uv_layers = uv_layers

        self.loss_rec = []

        # Initialize NNs
        if ExistModel == 0:
            self.uv_weights, self.uv_biases = self.initialize_NN(self.uv_layers)
        else:
            print("Loading uv NN ...")
            self.uv_weights, self.uv_biases = self.load_NN(uvDir, self.uv_layers)

        # tf placeholders
        self.learning_rate = tf.placeholder(tf.float32, shape=[])
        self.x_tf = tf.placeholder(tf.float32, shape=[None, self.x_c.shape[1]])
        self.y_tf = tf.placeholder(tf.float32, shape=[None, self.y_c.shape[1]])

        self.x_WALL_tf = tf.placeholder(tf.float32, shape=[None, self.x_WALL.shape[1]])
        self.y_WALL_tf = tf.placeholder(tf.float32, shape=[None, self.y_WALL.shape[1]])

        self.x_OUTLET_tf = tf.placeholder(tf.float32, shape=[None, self.x_OUTLET.shape[1]])
        self.y_OUTLET_tf = tf.placeholder(tf.float32, shape=[None, self.y_OUTLET.shape[1]])

        self.x_INLET_tf = tf.placeholder(tf.float32, shape=[None, self.x_INLET.shape[1]])
        self.y_INLET_tf = tf.placeholder(tf.float32, shape=[None, self.y_INLET.shape[1]])
        self.u_INLET_tf = tf.placeholder(tf.float32, shape=[None, self.u_INLET.shape[1]])
        self.v_INLET_tf = tf.placeholder(tf.float32, shape=[None, self.v_INLET.shape[1]])

        self.x_c_tf = tf.placeholder(tf.float32, shape=[None, self.x_c.shape[1]])
        self.y_c_tf = tf.placeholder(tf.float32, shape=[None, self.y_c.shape[1]])

        # tf graphs
        self.u_pred, self.v_pred, self.p_pred, _, _, _ = self.net_uv(self.x_tf, self.y_tf)
        self.f_pred_u, self.f_pred_v, self.f_pred_s11, self.f_pred_s22, self.f_pred_s12, \
            self.f_pred_p = self.net_f(self.x_c_tf, self.y_c_tf)
        self.u_WALL_pred, self.v_WALL_pred, _, _, _, _ = self.net_uv(self.x_WALL_tf, self.y_WALL_tf)
        self.u_INLET_pred, self.v_INLET_pred, _, _, _, _ = self.net_uv(self.x_INLET_tf, self.y_INLET_tf)
        _, _, self.p_OUTLET_pred, _, _, _ = self.net_uv(self.x_OUTLET_tf, self.y_OUTLET_tf)

        self.loss_f = tf.reduce_mean(tf.square(self.f_pred_u)) \
                      + tf.reduce_mean(tf.square(self.f_pred_v)) \
                      + tf.reduce_mean(tf.square(self.f_pred_s11)) \
                      + tf.reduce_mean(tf.square(self.f_pred_s22)) \
                      + tf.reduce_mean(tf.square(self.f_pred_s12)) \
                      + tf.reduce_mean(tf.square(self.f_pred_p))
        self.loss_WALL = tf.reduce_mean(tf.square(self.u_WALL_pred)) \
                         + tf.reduce_mean(tf.square(self.v_WALL_pred))
        self.loss_INLET = tf.reduce_mean(tf.square(self.u_INLET_pred - self.u_INLET_tf)) \
                          + tf.reduce_mean(tf.square(self.v_INLET_pred - self.v_INLET_tf))
        self.loss_OUTLET = tf.reduce_mean(tf.square(self.p_OUTLET_pred))

        self.loss = self.loss_f + 2 * (self.loss_WALL + self.loss_INLET + self.loss_OUTLET)

        # Optimizer for solution (replaces tf.contrib.opt.ScipyOptimizerInterface, same options)
        self.optimizer = ScipyLBFGSB(self.loss,
                                     var_list=self.uv_weights + self.uv_biases,
                                     options={'maxiter': bfgs_maxiter,
                                              'maxfun': bfgs_maxiter,
                                              'maxcor': 50,
                                              'maxls': 50,
                                              'ftol': 1 * np.finfo(float).eps})

        self.optimizer_Adam = tf.train.AdamOptimizer(learning_rate=self.learning_rate)
        self.train_op_Adam = self.optimizer_Adam.minimize(self.loss,
                                                          var_list=self.uv_weights + self.uv_biases)

        # tf session
        self.sess = tf.Session(config=tf.ConfigProto(allow_soft_placement=True,
                                                     log_device_placement=False))
        init = tf.global_variables_initializer()
        self.sess.run(init)

    def initialize_NN(self, layers):
        weights = []
        biases = []
        num_layers = len(layers)
        for l in range(0, num_layers - 1):
            W = self.xavier_init(size=[layers[l], layers[l + 1]])
            b = tf.Variable(tf.zeros([1, layers[l + 1]], dtype=tf.float32), dtype=tf.float32)
            weights.append(W)
            biases.append(b)
        return weights, biases

    def xavier_init(self, size):
        in_dim = size[0]
        out_dim = size[1]
        xavier_stddev = np.sqrt(2 / (in_dim + out_dim))
        return tf.Variable(tf.truncated_normal([in_dim, out_dim], stddev=xavier_stddev, dtype=tf.float32),
                           dtype=tf.float32)

    def save_NN(self, fileDir):
        uv_weights = self.sess.run(self.uv_weights)
        uv_biases = self.sess.run(self.uv_biases)
        with open(fileDir, 'wb') as f:
            pickle.dump([uv_weights, uv_biases], f)
            print("Save uv NN parameters successfully...")

    def load_NN(self, fileDir, layers):
        weights = []
        biases = []
        num_layers = len(layers)
        import warnings
        with open(fileDir, 'rb') as f, warnings.catch_warnings():
            warnings.simplefilter('ignore')   # pickle written with an old NumPy
            uv_weights, uv_biases = pickle.load(f)

            # Stored model must has the same # of layers
            assert num_layers == (len(uv_weights) + 1)

            for num in range(0, num_layers - 1):
                W = tf.Variable(uv_weights[num], dtype=tf.float32)
                b = tf.Variable(uv_biases[num], dtype=tf.float32)
                weights.append(W)
                biases.append(b)
            print(" - Load NN parameters successfully...")
        return weights, biases

    def neural_net(self, X, weights, biases):
        num_layers = len(weights) + 1
        H = X
        # H = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        for l in range(0, num_layers - 2):
            W = weights[l]
            b = biases[l]
            H = tf.tanh(tf.add(tf.matmul(H, W), b))
        W = weights[-1]
        b = biases[-1]
        Y = tf.add(tf.matmul(H, W), b)
        return Y

    def net_uv(self, x, y):
        psips = self.neural_net(tf.concat([x, y], 1), self.uv_weights, self.uv_biases)
        psi = psips[:, 0:1]
        p = psips[:, 1:2]
        s11 = psips[:, 2:3]
        s22 = psips[:, 3:4]
        s12 = psips[:, 4:5]
        u = tf.gradients(psi, y)[0]
        v = -tf.gradients(psi, x)[0]
        return u, v, p, s11, s22, s12

    def net_f(self, x, y):
        rho = self.rho
        mu = self.mu
        u, v, p, s11, s22, s12 = self.net_uv(x, y)

        s11_1 = tf.gradients(s11, x)[0]
        s12_2 = tf.gradients(s12, y)[0]
        s22_2 = tf.gradients(s22, y)[0]
        s12_1 = tf.gradients(s12, x)[0]

        # Plane stress problem
        u_x = tf.gradients(u, x)[0]
        u_y = tf.gradients(u, y)[0]

        v_x = tf.gradients(v, x)[0]
        v_y = tf.gradients(v, y)[0]

        # f_u:=Sxx_x+Sxy_y
        f_u = rho * (u * u_x + v * u_y) - s11_1 - s12_2
        f_v = rho * (u * v_x + v * v_y) - s12_1 - s22_2

        # f_mass = u_x+v_y (satisfied exactly by the stream function)

        f_s11 = -p + 2 * mu * u_x - s11
        f_s22 = -p + 2 * mu * v_y - s22
        f_s12 = mu * (u_y + v_x) - s12

        f_p = p + (s11 + s22) / 2

        return f_u, f_v, f_s11, f_s22, f_s12, f_p

    def callback(self, loss):
        self.count = self.count + 1
        self.loss_rec.append(loss)
        if self.count % 100 == 0:
            print('{} th iterations, Loss: {:.3e}'.format(self.count, loss))

    def _feed(self):
        return {self.x_c_tf: self.x_c, self.y_c_tf: self.y_c,
                self.x_WALL_tf: self.x_WALL, self.y_WALL_tf: self.y_WALL,
                self.x_INLET_tf: self.x_INLET, self.y_INLET_tf: self.y_INLET,
                self.u_INLET_tf: self.u_INLET, self.v_INLET_tf: self.v_INLET,
                self.x_OUTLET_tf: self.x_OUTLET, self.y_OUTLET_tf: self.y_OUTLET}

    def train(self, iter, learning_rate):
        tf_dict = self._feed()
        tf_dict[self.learning_rate] = learning_rate

        loss_WALL = []
        loss_f = []
        loss_INLET = []
        loss_OUTLET = []

        fetches = [self.train_op_Adam, self.loss, self.loss_WALL, self.loss_f, self.loss_INLET, self.loss_OUTLET]
        for it in range(iter):
            # one run: losses are those evaluated with the weights before this step
            _, l, lw, lf, li, lo = self.sess.run(fetches, tf_dict)

            if it % 100 == 0:
                print('It: %d, Loss: %.3e' % (it, l))

            loss_WALL.append(lw)
            loss_f.append(lf)
            self.loss_rec.append(l)
            loss_INLET.append(li)
            loss_OUTLET.append(lo)

        return loss_WALL, loss_INLET, loss_OUTLET, loss_f, self.loss

    def train_bfgs(self):
        return self.optimizer.minimize(self.sess, feed_dict=self._feed(), loss_callback=self.callback)

    def predict(self, x_star, y_star):
        u_star, v_star, p_star = self.sess.run([self.u_pred, self.v_pred, self.p_pred],
                                               {self.x_tf: x_star, self.y_tf: y_star})
        return u_star, v_star, p_star

    def getloss(self):
        loss_WALL, loss_INLET, loss_OUTLET, loss_f, loss = self.sess.run(
            [self.loss_WALL, self.loss_INLET, self.loss_OUTLET, self.loss_f, self.loss], self._feed())
        return loss_WALL, loss_INLET, loss_OUTLET, loss_f, loss

## 5. Helper functions

In [ ]:
def DelCylPT(XY_c, xc=0.0, yc=0.0, r=0.1):
    '''
    delete points within cylinder
    '''
    dst = np.sqrt((XY_c[:, 0] - xc) ** 2 + (XY_c[:, 1] - yc) ** 2)
    return XY_c[dst > r, :]


def preprocess(dir='FenicsSol.mat'):
    '''
    Load reference solution from Fenics or Fluent
    '''
    data = scipy.io.loadmat(dir)

    X = data['x']
    Y = data['y']
    P = data['p']
    vx = data['vx']
    vy = data['vy']

    x_star = X.flatten()[:, None]
    y_star = Y.flatten()[:, None]
    p_star = P.flatten()[:, None]
    vx_star = vx.flatten()[:, None]
    vy_star = vy.flatten()[:, None]

    return x_star, y_star, vx_star, vy_star, p_star


def postProcess(xmin, xmax, ymin, ymax, field_FLUENT, field_MIXED, s=2, alpha=0.5, marker='o',
                fname='./uvp.png'):
    '''
    Side-by-side scatter plots: mixed-form PINN (left) vs Fluent (right) for u, v, p.
    '''
    [x_FLUENT, y_FLUENT, u_FLUENT, v_FLUENT, p_FLUENT] = field_FLUENT
    [x_MIXED, y_MIXED, u_MIXED, v_MIXED, p_MIXED] = field_MIXED

    fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(10, 6))
    fig.subplots_adjust(hspace=0.25, wspace=0.2)

    rows = [(u_MIXED, u_FLUENT, r'$u$ (m/s)', {}),
            (v_MIXED, v_FLUENT, r'$v$ (m/s)', {}),
            (p_MIXED, p_FLUENT, 'Pressure (Pa)', dict(vmin=-0.25, vmax=4.0))]

    for i, (f_mix, f_flu, title, lims) in enumerate(rows):
        for j, (xx, yy, ff, a, lab) in enumerate([(x_MIXED, y_MIXED, f_mix, alpha - 0.1 if i < 2 else alpha, 'PINN'),
                                                  (x_FLUENT, y_FLUENT, f_flu, alpha, 'Fluent')]):
            cf = ax[i, j].scatter(xx, yy, c=ff, alpha=a, edgecolors='none', cmap='rainbow',
                                  marker=marker, s=int(s), **lims)
            ax[i, j].axis('square')
            for spine in ax[i, j].spines.values():
                spine.set_visible(False)
            ax[i, j].set_xticks([])
            ax[i, j].set_yticks([])
            ax[i, j].set_xlim([xmin, xmax])
            ax[i, j].set_ylim([ymin, ymax])
            ax[i, j].set_title(f'{lab}: {title}')
            fig.colorbar(cf, ax=ax[i, j], fraction=0.046, pad=0.04)

    plt.savefig(fname, dpi=300)
    plt.show()
    plt.close('all')


def multiple_formatter(denominator=2, number=np.pi, latex=r'\pi'):
    def gcd(a, b):
        while b:
            a, b = b, a % b
        return a

    def _multiple_formatter(x, pos):
        den = denominator
        num = int(np.rint(den * x / number))
        com = gcd(num, den)
        (num, den) = (int(num / com), int(den / com))
        if den == 1:
            if num == 0:
                return r'$0$'
            if num == 1:
                return r'$%s$' % latex
            elif num == -1:
                return r'$-%s$' % latex
            else:
                return r'$%s%s$' % (num, latex)
        else:
            if num == 1:
                return r'$\frac{%s}{%s}$' % (latex, den)
            elif num == -1:
                return r'$\frac{-%s}{%s}$' % (latex, den)
            else:
                return r'$\frac{%s%s}{%s}$' % (num, latex, den)
    return _multiple_formatter


class Multiple:
    def __init__(self, denominator=2, number=np.pi, latex=r'\pi'):
        self.denominator = denominator
        self.number = number
        self.latex = latex

    def locator(self):
        return plt.MultipleLocator(self.number / self.denominator)

    def formatter(self):
        return plt.FuncFormatter(multiple_formatter(self.denominator, self.number, self.latex))


def rel_l2(pred, ref):
    return np.linalg.norm(pred - ref) / np.linalg.norm(ref)


def compare_with_fluent(model, field_FLUENT):
    '''
    Relative L2 errors of the PINN at the Fluent nodes.
    p is compared after removing the mean, since the PINN only fixes p at the outlet.
    '''
    x_F, y_F, u_F, v_F, p_F = field_FLUENT
    u_P, v_P, p_P = model.predict(x_F, y_F)
    errs = {'u': rel_l2(u_P, u_F),
            'v': rel_l2(v_P, v_F),
            'p': rel_l2(p_P, p_F),
            'p (mean removed)': rel_l2(p_P - p_P.mean(), p_F - p_F.mean())}
    for k, e in errs.items():
        print(f'  relative L2 error {k:>17s}: {e:.3e}')
    return errs


def pinn_field(model):
    '''Mixed-form PINN prediction on a regular grid, with points inside the cylinder removed.'''
    x_PINN = np.linspace(0, 1.1, 251)
    y_PINN = np.linspace(0, 0.41, 101)
    x_PINN, y_PINN = np.meshgrid(x_PINN, y_PINN)
    x_PINN = x_PINN.flatten()[:, None]
    y_PINN = y_PINN.flatten()[:, None]
    dst = ((x_PINN - 0.2) ** 2 + (y_PINN - 0.2) ** 2) ** 0.5
    x_PINN = x_PINN[dst >= 0.05].flatten()[:, None]
    y_PINN = y_PINN[dst >= 0.05].flatten()[:, None]
    u_PINN, v_PINN, p_PINN = model.predict(x_PINN, y_PINN)
    return [x_PINN, y_PINN, u_PINN, v_PINN, p_PINN]

## 6. Geometry, boundary points and collocation points

In [ ]:
# Domain bounds
lb = np.array([0, 0])
ub = np.array([1.1, 0.41])

# Network configuration
uv_layers = [2] + 8 * [40] + [5]

# WALL = [x, y], u=v=0
wall_up = [0.0, 0.41] + [1.1, 0.0] * lhs(2, 441)
wall_lw = [0.0, 0.00] + [1.1, 0.0] * lhs(2, 441)

# INLET = [x, y, u, v]
U_max = 1.0
INLET = [0.0, 0.0] + [0.0, 0.41] * lhs(2, 201)
y_INLET = INLET[:, 1:2]
u_INLET = 4 * U_max * y_INLET * (0.41 - y_INLET) / (0.41 ** 2)
v_INLET = 0 * y_INLET
INLET = np.concatenate((INLET, u_INLET, v_INLET), 1)

# OUTLET = [x, y], p=0
OUTLET = [1.1, 0.0] + [0.0, 0.41] * lhs(2, 201)

# Cylinder surface
r = 0.05
theta = [0.0] + [2 * np.pi] * lhs(1, 251)
x_CYLD = np.multiply(r, np.cos(theta)) + 0.2
y_CYLD = np.multiply(r, np.sin(theta)) + 0.2
CYLD = np.concatenate((x_CYLD, y_CYLD), 1)

WALL = np.concatenate((CYLD, wall_up, wall_lw), 0)

# Collocation point for equation residual (refined around the cylinder)
XY_c = lb + (ub - lb) * lhs(2, 40000)
XY_c_refine = [0.1, 0.1] + [0.2, 0.2] * lhs(2, 10000)
XY_c = np.concatenate((XY_c, XY_c_refine), 0)
XY_c = DelCylPT(XY_c, xc=0.2, yc=0.2, r=0.05)

XY_c = np.concatenate((XY_c, WALL, CYLD, OUTLET, INLET[:, 0:2]), 0)

print('Collocation points:', XY_c.shape)

In [ ]:
# Visualize the collocation points
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.set_aspect('equal')
ax.scatter(XY_c[:, 0:1], XY_c[:, 1:2], marker='o', s=1, alpha=0.1, color='blue', label='collocation')
ax.scatter(WALL[:, 0:1], WALL[:, 1:2], marker='o', s=4, alpha=0.4, color='green', label='wall + cylinder')
ax.scatter(OUTLET[:, 0:1], OUTLET[:, 1:2], marker='o', s=4, alpha=0.4, color='orange', label='outlet')
ax.scatter(INLET[:, 0:1], INLET[:, 1:2], marker='o', s=4, alpha=0.4, color='red', label='inlet')
ax.legend(loc='upper right', markerscale=4, fontsize=8)
plt.show()

## 7. Build the model and evaluate the pretrained network

In [ ]:
if LOAD_PRETRAINED:
    model = PINN_laminar_flow(XY_c, INLET, OUTLET, WALL, uv_layers, lb, ub,
                              ExistModel=1, uvDir=PRETRAINED_FILE, bfgs_maxiter=BFGS_MAXITER)
else:
    model = PINN_laminar_flow(XY_c, INLET, OUTLET, WALL, uv_layers, lb, ub, bfgs_maxiter=BFGS_MAXITER)

l_wall, l_in, l_out, l_f, l_tot = model.getloss()
print(f'Initial loss: total {l_tot:.3e} | PDE {l_f:.3e} | wall {l_wall:.3e} | inlet {l_in:.3e} | outlet {l_out:.3e}')

# Load Fluent reference
field_FLUENT = list(preprocess(dir=FLUENT_FILE))
print('Fluent nodes:', field_FLUENT[0].shape[0])

print('Before training:')
errs_before = compare_with_fluent(model, field_FLUENT)

In [ ]:
postProcess(xmin=0, xmax=1.1, ymin=0, ymax=0.41, field_FLUENT=field_FLUENT,
            field_MIXED=pinn_field(model), s=3, alpha=0.5, fname='./uvp_before.png')

## 8. Training: Adam, then L-BFGS-B

In [ ]:
if RUN_TRAINING:
    start_time = time.time()
    loss_WALL, loss_INLET, loss_OUTLET, loss_f, _ = model.train(iter=ADAM_ITERS, learning_rate=ADAM_LR)
    print("--- Adam: %.1f seconds ---" % (time.time() - start_time))
    t1 = time.time()
    if BFGS_MAXITER > 0:
        model.train_bfgs()
    print("--- L-BFGS-B: %.1f seconds ---" % (time.time() - t1))
    print("--- total: %.1f seconds ---" % (time.time() - start_time))

    # Save neural network and loss history
    model.save_NN(SAVE_FILE)
    with open('loss_history.pickle', 'wb') as f:
        pickle.dump(model.loss_rec, f)

In [ ]:
if RUN_TRAINING and len(model.loss_rec) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(model.loss_rec, lw=1)
    if ADAM_ITERS > 0:
        ax.axvline(ADAM_ITERS, color='k', ls='--', lw=0.8)
        ax.text(ADAM_ITERS, ax.get_ylim()[1], ' L-BFGS-B →', va='top', fontsize=9)
    ax.set_xlabel('iteration')
    ax.set_ylabel('total loss')
    ax.set_title('Loss history')
    plt.show()

## 9. Results after training

In [ ]:
l_wall, l_in, l_out, l_f, l_tot = model.getloss()
print(f'Final loss: total {l_tot:.3e} | PDE {l_f:.3e} | wall {l_wall:.3e} | inlet {l_in:.3e} | outlet {l_out:.3e}')
print('After training:')
errs_after = compare_with_fluent(model, field_FLUENT)

# Plot the comparison of u, v, p
postProcess(xmin=0, xmax=1.1, ymin=0, ymax=0.41, field_FLUENT=field_FLUENT,
            field_MIXED=pinn_field(model), s=3, alpha=0.5, fname='./uvp.png')